# Проверка на своих картинках

У модели по одной голове на семейство генераторов: OpenAI, Google, Flux и так далее. Каждая голова отвечает на
свой вопрос: эту картинку сделало моё семейство или она реальная. Балл картинки — самая уверенная голова.
Порог берётся из `report.md`, его пишет `scripts/run_test.py`: выше порога около 1 % реальных картинок.
Без отчёта порог 0.5.

In [ ]:
import os, io, re, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.model import Detector

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
size = cfg['train']['size']   # 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)
families = model.config['families']

# порог из отчёта теста; без отчёта 0.5
threshold = 0.5
if (out / 'report.md').exists():
    threshold = float(re.search(r'threshold ([\d.]+)', (out / 'report.md').read_text()).group(1))
threshold

Картинка готовится как на тесте: не-JPEG один раз сжимается в JPEG с качеством 90 (на обучении так сжимались
все не-JPEG), потом картинка целиком сжимается до 512×512.

In [ ]:
def prepare(path):
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)   # [3, 512, 512], от 0 до 1


@torch.no_grad()
def predict(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        logits = model(x)
    return torch.sigmoid(logits[0].float()).cpu().numpy()   # балл каждой головы

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

probs = predict(path)
prob, family = probs.max(), families[probs.argmax()]
verdict = f'сгенерирована, похоже на {family}' if prob > threshold else 'реальная'
print(f'{verdict}: балл {prob:.3f}, порог {threshold:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(Image.open(path).convert('RGB'))
ax[0].set_title(f'{verdict}: {prob:.3f}')
ax[0].axis('off')
ax[1].barh(families, probs)
ax[1].axvline(threshold, color='red', linestyle='--')
ax[1].set_xlim(0, 1)
ax[1].invert_yaxis()
ax[1].set_title('баллы голов, красная линия — порог')
plt.tight_layout()
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')   # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        probs = predict(p)
        print(f"{probs.max():.3f}  {'FAKE' if probs.max() > threshold else 'real'}  {families[probs.argmax()]:10s}  {p.name}")